# evcxr-lab: getting started

Smoke test for the Rust kernel. Run all cells; if the last one renders a table, the lab works.

Useful kernel commands: `:help`, `:vars`, `:dep`, `:timing`, `:opt`, `:clear`.

In [ ]:
:version

Values are evaluated and the last expression of a cell is displayed.

In [ ]:
let samples: Vec<f64> = (1..=10).map(|n| n as f64).collect();
let mean = samples.iter().sum::<f64>() / samples.len() as f64;
mean

Variables, types and functions persist across cells.

In [ ]:
pub fn variance(xs: &[f64], mean: f64) -> f64 {
    xs.iter().map(|x| (x - mean).powi(2)).sum::<f64>() / xs.len() as f64
}

variance(&samples, mean).sqrt()

## Crates from crates.io

`:dep` compiles a dependency into the session (the first one takes a few seconds). Crates with
a relative `path` resolve against the notebook's directory, so a crate living in this repo can be
loaded with e.g. `:dep mylib = { path = "../crates/mylib" }`.

In [ ]:
:dep rand = "0.9"

use rand::{Rng, SeedableRng};
use rand::rngs::StdRng;

let mut rng = StdRng::seed_from_u64(42);
let draws: Vec<u32> = (0..8).map(|_| rng.random_range(1..=6)).collect();
draws

## Rich output

A type with an `evcxr_display` method can emit any mime type, so results can render as HTML,
SVG or images instead of `Debug` text.

In [ ]:
pub struct Histogram {
    pub counts: Vec<(u32, usize)>,
}

impl Histogram {
    pub fn evcxr_display(&self) {
        let mut html = String::from("<table><tr><th>value</th><th>count</th></tr>");
        for (value, count) in &self.counts {
            html.push_str(&format!(
                "<tr><td>{value}</td><td>{}</td></tr>",
                "\u{2588}".repeat(*count)
            ));
        }
        html.push_str("</table>");
        println!("EVCXR_BEGIN_CONTENT text/html\n{html}\nEVCXR_END_CONTENT");
    }
}

let mut counts: Vec<(u32, usize)> = (1..=6)
    .map(|face| (face, draws.iter().filter(|d| **d == face).count()))
    .collect();
counts.sort_by_key(|(face, _)| *face);

// Note: bind the value, then name it. Moving a variable that the kernel is
// trying to persist (here `counts`) directly into the final expression makes
// evcxr fall back to `Debug` formatting instead of `evcxr_display`.
let histogram = Histogram { counts };
histogram